# Water-filling profiles for every JBB target

**Input:** `task_alphabet_150_scored.csv`, with 100 JBB rows and 50 SuperNI rows.

This notebook uses the reference population prior:

\[
\rho
=0.10\,\overline{\epsilon}_{\mathrm{JBB}}
+0.90\,\overline{\epsilon}_{\mathrm{SuperNI}}.
\]

For each JBB target \(t_b\), it treats the 50 SuperNI tasks as fractional auxiliaries:

\[
r_b(x)=
\frac{\epsilon_b+\sum_{j=1}^{50}x_j\epsilon_j}
{1+\sum_{j=1}^{50}x_j},
\qquad 0\le x_j\le 1.
\]

It then computes:

\[
\min_x\sum_jx_j \quad\text{s.t.}\quad r_b(x)=\rho,
\]

and the auxiliary-budget curve:

\[
D_b(B)
=
\min_{\sum_jx_j\le B}
|\rho-r_b(x)|,
\qquad B=0,1,\ldots,50.
\]

This is a theory-side notebook only. It does not generate prompts or run downstream inference.


## Modeling convention

The code uses the **unweighted bundle average**:
every component included in a bundle has equal weight in \(r_b(x)\).

The 10% / 90% population assumption is used to define \(\rho\). The fractional variables represent auxiliary-task mass.


## 0. Load the scored 150-task dictionary


## Repository paths

Run this setup cell first. Inputs, outputs, and runtime caches use separate repository directories.


In [ ]:
from pathlib import Path
import os

# Launch Jupyter from the repository, or set INTENT_REPO_ROOT explicitly.
_start = Path(os.environ.get("INTENT_REPO_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next(
    (p for p in (_start, *_start.parents)
     if (p / "data/artifact_manifest.json").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Clone/extract the repository and set INTENT_REPO_ROOT to its root.")
DATA_DIR = Path(os.environ.get("INTENT_DATA_DIR", REPO_ROOT / "data")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("INTENT_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
WORK_DIR = Path(os.environ.get("INTENT_WORK_DIR", REPO_ROOT / ".work")).expanduser().resolve()
for _directory in (DATA_DIR, OUTPUT_ROOT, WORK_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

# Colab uploads are optional; local runs use configured input paths.
try:
    from google.colab import files as notebook_files
except ImportError:
    class LocalFiles:
        @staticmethod
        def upload():
            raise FileNotFoundError("Set the notebook input path to an existing local file.")

        @staticmethod
        def download(filename):
            print("Saved:", Path(filename).resolve())

    notebook_files = LocalFiles()


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_CSV = (DATA_DIR / "task_alphabet_150_scored.csv")
OUTPUT_DIR = (OUTPUT_ROOT / "waterfilling_profiles")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BAD_POPULATION_MASS = 0.10
GOOD_POPULATION_MASS = 0.90
TOL = 1e-10

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Upload your file to {INPUT_CSV}. "
        "In Colab, use the Files panel or files.upload()."
    )

task_df = pd.read_csv(INPUT_CSV)

required = {"task_id", "source", "epsilon"}
missing = required.difference(task_df.columns)
if missing:
    raise ValueError(f"Input CSV is missing columns: {sorted(missing)}")

task_df["epsilon"] = pd.to_numeric(task_df["epsilon"], errors="raise")
if not task_df["epsilon"].between(0.0, 1.0).all():
    raise ValueError("All epsilon values must lie in [0, 1].")

jbb_df = task_df[task_df["source"].astype(str).str.lower() == "jbb"].copy()
aux_df = task_df[task_df["source"].astype(str).str.lower() == "superni"].copy()

if len(jbb_df) != 100:
    raise ValueError(f"Expected 100 JBB rows, found {len(jbb_df)}.")
if len(aux_df) != 50:
    raise ValueError(f"Expected 50 SuperNI rows, found {len(aux_df)}.")

print(f"Loaded {len(jbb_df)} JBB targets and {len(aux_df)} SuperNI auxiliaries.")


## 1. Compute the 10% / 90% population prior

\[
\rho
=
0.10\left(\frac{1}{100}\sum_{b\in\mathrm{JBB}}\epsilon_b\right)
+
0.90\left(\frac{1}{50}\sum_{j\in\mathrm{SuperNI}}\epsilon_j\right).
\]


In [ ]:
mean_jbb_epsilon = float(jbb_df["epsilon"].mean())
mean_superni_epsilon = float(aux_df["epsilon"].mean())

rho = (
    BAD_POPULATION_MASS * mean_jbb_epsilon
    + GOOD_POPULATION_MASS * mean_superni_epsilon
)

prior_summary = pd.DataFrame([{
    "mean_jbb_epsilon": mean_jbb_epsilon,
    "mean_superni_epsilon": mean_superni_epsilon,
    "bad_population_mass": BAD_POPULATION_MASS,
    "good_population_mass": GOOD_POPULATION_MASS,
    "rho": rho,
}])

display(prior_summary)
print(f"rho = {rho:.12f}")


## 2. Water-filling functions

If \(\epsilon_b>\rho\), use SuperNI auxiliaries with \(\epsilon_j<\rho\),
in ascending order of \(\epsilon_j\). This is the water-filling order.

The code also handles the symmetric \(\epsilon_b<\rho\) case.


In [ ]:
aux_eps = aux_df["epsilon"].to_numpy(dtype=float)
aux_ids = aux_df["task_id"].astype(str).to_numpy()

def bundle_score(epsilon_b, epsilon_aux, x):
    x = np.asarray(x, dtype=float)
    return float((epsilon_b + np.dot(x, epsilon_aux)) / (1.0 + x.sum()))

def useful_order(epsilon_b, epsilon_aux, rho, tol=TOL):
    delta = float(epsilon_b - rho)
    if abs(delta) <= tol:
        return delta, np.array([], dtype=int), np.array([], dtype=float)

    signed_gains = np.sign(delta) * (rho - np.asarray(epsilon_aux, dtype=float))
    useful = np.flatnonzero(signed_gains > tol)
    order = useful[np.argsort(-signed_gains[useful])]
    return delta, order, signed_gains

def waterfill_solution(epsilon_b, epsilon_aux, rho, tol=TOL):
    epsilon_aux = np.asarray(epsilon_aux, dtype=float)
    x = np.zeros_like(epsilon_aux)
    delta, order, gains = useful_order(epsilon_b, epsilon_aux, rho, tol)

    if abs(delta) <= tol:
        return x, "target_already_matches_rho"

    remaining = abs(delta)
    for j in order:
        take = min(1.0, remaining / gains[j])
        x[j] = take
        remaining -= take * gains[j]
        if remaining <= tol:
            break

    r_x = bundle_score(epsilon_b, epsilon_aux, x)
    status = "exact_match" if abs(r_x - rho) <= 1e-8 else "insufficient_auxiliary_capacity"
    return x, status

def waterfill_under_budget(epsilon_b, epsilon_aux, rho, budget, tol=TOL):
    epsilon_aux = np.asarray(epsilon_aux, dtype=float)
    x = np.zeros_like(epsilon_aux)
    delta, order, gains = useful_order(epsilon_b, epsilon_aux, rho, tol)

    if abs(delta) <= tol or budget <= tol:
        return x, bundle_score(epsilon_b, epsilon_aux, x)

    remaining_deficit = abs(delta)
    remaining_budget = float(budget)

    for j in order:
        if remaining_budget <= tol or remaining_deficit <= tol:
            break
        take = min(
            1.0,
            remaining_budget,
            remaining_deficit / gains[j],
        )
        x[j] = take
        remaining_budget -= take
        remaining_deficit -= take * gains[j]

    return x, bundle_score(epsilon_b, epsilon_aux, x)

def waterfill_breakpoints(epsilon_b, epsilon_aux, rho, tol=TOL):
    epsilon_aux = np.asarray(epsilon_aux, dtype=float)
    x = np.zeros_like(epsilon_aux)
    rows = []

    def append_row(step, last_aux_index, last_aux_inclusion):
        r_x = bundle_score(epsilon_b, epsilon_aux, x)
        rows.append({
            "path_step": step,
            "auxiliary_mass_used": float(x.sum()),
            "expected_bundle_size": float(1.0 + x.sum()),
            "r_x": r_x,
            "abs_rho_minus_r_x": abs(r_x - rho),
            "last_aux_index": last_aux_index,
            "last_aux_inclusion": last_aux_inclusion,
        })

    append_row(0, None, 0.0)

    delta, order, gains = useful_order(epsilon_b, epsilon_aux, rho, tol)
    if abs(delta) <= tol:
        return rows

    remaining = abs(delta)
    step = 0
    for j in order:
        take = min(1.0, remaining / gains[j])
        x[j] = take
        remaining -= take * gains[j]
        step += 1
        append_row(step, int(j), float(take))
        if remaining <= tol:
            break

    return rows


## 3. Run water filling for all 100 JBB targets


In [ ]:
solution_rows = []
selection_rows = []
breakpoint_rows = []
curve_rows = []

for _, target in jbb_df.iterrows():
    target_id = str(target["task_id"])
    epsilon_b = float(target["epsilon"])

    x_star, status = waterfill_solution(epsilon_b, aux_eps, rho)

    r_star = bundle_score(epsilon_b, aux_eps, x_star)
    solution_rows.append({
        "target_id": target_id,
        "behavior": target.get("behavior", pd.NA),
        "category": target.get("category", pd.NA),
        "epsilon_b": epsilon_b,
        "rho": rho,
        "status": status,
        "required_auxiliary_mass": float(x_star.sum()),
        "expected_bundle_size": float(1.0 + x_star.sum()),
        "r_x_at_solution": r_star,
        "abs_rho_minus_r_x_at_solution": abs(r_star - rho),
        "n_fully_included_auxiliaries": int((x_star >= 1.0 - TOL).sum()),
        "n_fractional_auxiliaries": int(((x_star > TOL) & (x_star < 1.0 - TOL)).sum()),
    })

    for j in np.flatnonzero(x_star > TOL):
        selection_rows.append({
            "target_id": target_id,
            "aux_task_id": aux_ids[j],
            "aux_epsilon": float(aux_eps[j]),
            "inclusion_level": float(x_star[j]),
            "fully_included": bool(x_star[j] >= 1.0 - TOL),
        })

    for row in waterfill_breakpoints(epsilon_b, aux_eps, rho):
        j = row.pop("last_aux_index")
        row.update({
            "target_id": target_id,
            "epsilon_b": epsilon_b,
            "rho": rho,
            "last_aux_task_id": aux_ids[j] if j is not None else pd.NA,
        })
        breakpoint_rows.append(row)

    for budget in range(len(aux_eps) + 1):
        x_budget, r_budget = waterfill_under_budget(
            epsilon_b, aux_eps, rho, budget
        )
        curve_rows.append({
            "target_id": target_id,
            "behavior": target.get("behavior", pd.NA),
            "category": target.get("category", pd.NA),
            "epsilon_b": epsilon_b,
            "rho": rho,
            "max_auxiliary_budget": budget,
            "actual_auxiliary_mass_used": float(x_budget.sum()),
            "expected_bundle_size_used": float(1.0 + x_budget.sum()),
            "r_x": r_budget,
            "abs_rho_minus_r_x": abs(r_budget - rho),
        })

solutions_df = pd.DataFrame(solution_rows)
selections_df = pd.DataFrame(selection_rows)
breakpoints_df = pd.DataFrame(breakpoint_rows)
curves_df = pd.DataFrame(curve_rows)

display(solutions_df.head())

not_exact = solutions_df[solutions_df["status"] != "exact_match"]
if len(not_exact):
    print("Targets without an exact fractional match:")
    display(not_exact)
else:
    print("All 100 targets achieve r(x)=rho under fractional water filling.")


## 4. Save per-target solutions and curves


In [ ]:
solutions_path = OUTPUT_DIR / "waterfilling_solutions_all_jbb.csv"
selections_path = OUTPUT_DIR / "waterfilling_selected_auxiliaries_all_jbb.csv"
breakpoints_path = OUTPUT_DIR / "waterfilling_continuous_paths_all_jbb.csv"
curves_path = OUTPUT_DIR / "waterfilling_abs_gap_curves_all_jbb.csv"

solutions_df.to_csv(solutions_path, index=False)
selections_df.to_csv(selections_path, index=False)
breakpoints_df.to_csv(breakpoints_path, index=False)
curves_df.to_csv(curves_path, index=False)

run_summary = {
    "input_csv": str(INPUT_CSV),
    "n_jbb_targets": int(len(jbb_df)),
    "n_superni_auxiliaries": int(len(aux_df)),
    "bad_population_mass": BAD_POPULATION_MASS,
    "good_population_mass": GOOD_POPULATION_MASS,
    "mean_jbb_epsilon": mean_jbb_epsilon,
    "mean_superni_epsilon": mean_superni_epsilon,
    "rho": rho,
    "bundle_score": "(epsilon_b + sum_j x_j epsilon_j) / (1 + sum_j x_j)",
    "water_filling": "fractional x_j in [0,1], minimizing sum_j x_j",
}
summary_path = OUTPUT_DIR / "waterfilling_run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2))

for path in [solutions_path, selections_path, breakpoints_path, curves_path, summary_path]:
    print(path)


## 5. Plot \(|\rho-r_b(x)|\) for every JBB target

The horizontal axis is the maximum auxiliary budget \(B\). The curve becomes zero once the exact fractional water-filling solution is affordable.


In [ ]:
plt.figure(figsize=(8, 5))
for _, group in curves_df.groupby("target_id", sort=False):
    plt.plot(
        group["max_auxiliary_budget"],
        group["abs_rho_minus_r_x"],
        alpha=0.25,
    )
plt.xlabel("Maximum auxiliary-task budget B")
plt.ylabel(r"$|\rho-r_b(x)|$")
plt.title("Fractional water-filling deviation curves: all JBB targets")
plt.grid(True, alpha=0.3)
plt.tight_layout()
all_curves_plot = OUTPUT_DIR / "waterfilling_abs_gap_all_targets.png"
plt.savefig(all_curves_plot, dpi=200)
plt.show()

aggregate_df = (
    curves_df.groupby("max_auxiliary_budget")["abs_rho_minus_r_x"]
    .agg(
        mean="mean",
        median="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
    )
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.plot(
    aggregate_df["max_auxiliary_budget"],
    aggregate_df["mean"],
    label="mean",
)
plt.plot(
    aggregate_df["max_auxiliary_budget"],
    aggregate_df["median"],
    label="median",
)
plt.fill_between(
    aggregate_df["max_auxiliary_budget"],
    aggregate_df["q25"],
    aggregate_df["q75"],
    alpha=0.2,
    label="interquartile range",
)
plt.xlabel("Maximum auxiliary-task budget B")
plt.ylabel(r"$|\rho-r_b(x)|$")
plt.title("Aggregate fractional water-filling deviation curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
aggregate_plot = OUTPUT_DIR / "waterfilling_abs_gap_aggregate.png"
plt.savefig(aggregate_plot, dpi=200)
plt.show()

aggregate_path = OUTPUT_DIR / "waterfilling_abs_gap_aggregate.csv"
aggregate_df.to_csv(aggregate_path, index=False)

print(all_curves_plot)
print(aggregate_plot)
print(aggregate_path)
display(aggregate_df.head(10))


## 6. Inspect fractional bundle sizes

`required_auxiliary_mass = 1.7` means the water-filling relaxation fully includes one auxiliary and includes the next one at level 0.7.


In [ ]:
display(
    solutions_df[
        [
            "target_id",
            "category",
            "epsilon_b",
            "rho",
            "required_auxiliary_mass",
            "expected_bundle_size",
            "n_fully_included_auxiliaries",
            "n_fractional_auxiliaries",
            "status",
        ]
    ].sort_values("required_auxiliary_mass", ascending=False)
)

print(solutions_df["required_auxiliary_mass"].describe())


In [ ]:
subset_df = curves_df[curves_df["max_auxiliary_budget"] <= 10]
avg_bundle_size = subset_df.groupby("max_auxiliary_budget")["expected_bundle_size_used"].mean()

plt.figure(figsize=(8, 5))
plt.plot(avg_bundle_size.index, avg_bundle_size.values, marker='o')
plt.xlabel("Maximum auxiliary-task budget B")
plt.ylabel("Average Expected Bundle Size")
plt.title("Average Expected Bundle Size vs Budget (up to B=10)")
plt.grid(True, alpha=0.3)
plt.xticks(range(11))
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
subset_curves = curves_df[curves_df["max_auxiliary_budget"] <= 10].copy()
subset_curves["rx_minus_rho"] = subset_curves["r_x"] - subset_curves["rho"]
avg_rx_minus_rho = subset_curves.groupby("max_auxiliary_budget")["rx_minus_rho"].mean()

plt.plot(avg_rx_minus_rho.index, avg_rx_minus_rho.values, marker='o')
plt.xlabel("Maximum auxiliary-task budget B")
plt.ylabel(r"Average $r_b(x) - \rho$")
plt.title("Average signed fractional water-filling deviation (up to B=10)")
plt.grid(True, alpha=0.3)
plt.xticks(range(11))
plt.tight_layout()
plt.show()
